In [2]:
import onnxruntime as rt
import tensorflow as tf

In [3]:
model_onnx_path = 'mnist.onnx'

In [4]:
providers = ['CPUExecutionProvider']
sess = rt.InferenceSession(model_onnx_path,providers=providers)

In [5]:
######## just for test
from keras.datasets import mnist
(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_tarin = (x_train / 255).astype('float32')
x_test = (x_test / 255).astype('float32')

In [6]:
# when we built the model defined the input as (28,28,1) but the original shape is (28,28)
# so we need to change the test shape to the shape which is accepted by the model!!
x_test = x_test.reshape(10000,28,28,1)
predict_onnx = sess.run(['dense'],{'args_0':x_test[:1000]})

In [7]:
predict_onnx[0].shape 

(1000, 10)

In [8]:
model_keras = tf.keras.models.load_model('mnist.keras')
predict_keras = model_keras.predict(x_test[:1000])

I0000 00:00:1782603431.477586    7834 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7060 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:01:00.0, compute capability: 6.1
I0000 00:00:1782603432.435332    7899 service.cc:148] XLA service 0x7d5a000032c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782603432.435365    7899 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce GTX 1070, Compute Capability 6.1
2026-06-28 01:37:12.443549: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782603432.475448    7899 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-06-28 01:37:12.709071: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{} for conv (f32[32,32,28,28]{3,2,1,0}, u8[0]{0}) c

24/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

I0000 00:00:1782603433.344399    7899 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-28 01:37:13.590857: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{} for conv (f32[8,32,28,28]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,32,28,28]{3,2,1,0}, f32[32,32,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]}
2026-06-28 01:37:13.642267: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:557] Omitted potentially buggy algorithm eng14{} for conv (f32[8,128,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,32,14,14]{3,2,1,0},

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step


In [9]:
#### to check if both values are same!
import numpy as np

In [10]:
is_close = np.isclose(predict_onnx[0],predict_keras,rtol=1e-6)

In [11]:
is_close.sum() / (len(is_close)*10)

np.float64(1.0)

In [12]:
is_close = np.isclose(predict_onnx[0],predict_keras,rtol=1e-7)

In [13]:
is_close.sum() / (len(is_close)*10)

np.float64(0.9309)

In [14]:
## the diffrence is so small!!

In [15]:
%%timeit
predict_onnx = sess.run(['dense'],{'args_0':x_test[:1000]})

667 ms ± 13.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
%%timeit
predict_keras = model_keras.predict(x_test[:1000])

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━